In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt

from vip_slap2_analysis.glutamate import summary as gs

sns.set()
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib notebook

In [ ]:
basepath = r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
summary_path = glob.glob(os.path.join(basepath,'**summary.xlsx'))[0]
summary_df = pd.read_excel(summary_path,sheet_name='subjects')
session_df = pd.read_excel(summary_path,sheet_name='sessions')

In [ ]:
target_mice = [
    803496,804730,804733,810196,
    809047,803121,
    826033,834788,838410]

In [ ]:
for mouse in target_mice:
    try:
        session_paths = [Path(p) for p in session_df[(session_df['subject_id']==mouse)
                                                     &(session_df['session_type']!='expression_check')]['session_dir'].values]
    except:
        print(f'No session path for {session_df[(session_df["subject_id"]==mouse)]["session_id"].values}')
    print(mouse)
    for sess in session_paths:
        summary_paths = glob.glob(os.path.join(sess, '**', 'SummaryLoCo**.mat'), recursive=True) 
    
        if len(summary_paths) == 0:
            print(f'No Summary_LoCo.mat found in {sess}')
            print('')
            continue

        summary_path = max(summary_paths, key=os.path.getmtime)
        exp = gs.GlutamateSummary(summary_path)
        n_trials = exp.n_trials
        print(f'Successfully loaded data from {sess.stem}')
        dmd1_valid_trials = exp.valid_trials[0]
        dmd2_valid_trials = exp.valid_trials[1]
        
        dmd1_vt_pct = len(dmd1_valid_trials)/n_trials
        dmd2_vt_pct = len(dmd2_valid_trials)/n_trials
        
        dmd_pct = [dmd1_vt_pct,dmd2_vt_pct]
        
        for i,dmd in enumerate([1,2]):
            print(f'% valid trials on DMD{dmd}: {dmd_pct[i]*100:.4}% with {exp.n_synapses[i]} synapses recorded')
        print('')
    print('')
#         print(f'Most recent Summary_LoCo.mat: {most_recent_summary}')
#         print('')

In [ ]:
# Summarize synapse yield by recording session day and imaging depth.
#
# Uses the same access pattern as the QC loop above:
#   1. Use `session_df` from summary.xlsx / sessions.
#   2. Restrict to `target_mice` and recording session_# 2-7.
#   3. Load the most recent SummaryLoCo*.mat for each session.
#   4. Read the number of synapses recorded on DMD1/DMD2 and map each DMD to
#      its depth using dmd1_depth / dmd2_depth from the sessions table.

session_day_col = 'session_#'
recording_days = range(2, 8)

target_session_df = session_df[
    session_df['subject_id'].isin(target_mice)
    & pd.to_numeric(session_df[session_day_col], errors='coerce').isin(recording_days)
].copy()

target_session_df[session_day_col] = pd.to_numeric(
    target_session_df[session_day_col], errors='coerce'
).astype('Int64')

session_depth_rows = []
missing_summary_rows = []

for _, sess_row in target_session_df.sort_values(['subject_id', session_day_col]).iterrows():
    mouse = int(sess_row['subject_id'])
    session_id = sess_row['session_id']
    session_day = int(sess_row[session_day_col])
    session_type = sess_row.get('session_type', None)
    session_dir = sess_row.get('session_dir', None)

    if pd.isna(session_dir):
        missing_summary_rows.append({
            'subject_id': mouse,
            'session_id': session_id,
            'session_#': session_day,
            'session_dir': session_dir,
            'reason': 'missing session_dir in summary.xlsx',
        })
        continue

    summary_paths = glob.glob(
        os.path.join(str(session_dir), '**', 'SummaryLoCo**.mat'),
        recursive=True,
    )

    if len(summary_paths) == 0:
        missing_summary_rows.append({
            'subject_id': mouse,
            'session_id': session_id,
            'session_#': session_day,
            'session_dir': session_dir,
            'reason': 'no SummaryLoCo*.mat found',
        })
        continue

    summary_loco_path = max(summary_paths, key=os.path.getmtime)

    try:
        exp = gs.GlutamateSummary(summary_loco_path)
    except Exception as exc:
        missing_summary_rows.append({
            'subject_id': mouse,
            'session_id': session_id,
            'session_#': session_day,
            'session_dir': session_dir,
            'reason': f'GlutamateSummary load failed: {exc}',
        })
        continue

    for dmd in (1, 2):
        depth_col = f'dmd{dmd}_depth'
        depth_um = sess_row.get(depth_col, np.nan)

        try:
            n_synapses = exp.n_synapses[dmd - 1]
        except (IndexError, TypeError):
            n_synapses = np.nan

        session_depth_rows.append({
            'subject_id': mouse,
            'session_id': session_id,
            'session_#': session_day,
            'session_type': session_type,
            'dmd': dmd,
            'depth_um': depth_um,
            'n_synapses': n_synapses,
            'summary_loco_path': summary_loco_path,
        })

session_depth_counts = pd.DataFrame(session_depth_rows)
missing_summary_df = pd.DataFrame(missing_summary_rows)

if session_depth_counts.empty:
    print('No SummaryLoCo files were loaded for the target mice / session_# 2-7 filter.')
else:
    session_depth_counts['depth_um'] = pd.to_numeric(
        session_depth_counts['depth_um'], errors='coerce'
    )
    session_depth_counts['n_synapses'] = pd.to_numeric(
        session_depth_counts['n_synapses'], errors='coerce'
    )

    session_depth_summary = (
        session_depth_counts
        .dropna(subset=['depth_um'])
        .groupby(['session_#', 'depth_um'], as_index=False)
        .agg(
            n_synapses=('n_synapses', 'sum'),
            n_mice=('subject_id', 'nunique'),
            n_sessions=('session_id', 'nunique'),
            n_dmds=('dmd', 'count'),
        )
        .sort_values(['session_#', 'depth_um'])
    )

    synapse_count_pivot = (
        session_depth_summary
        .pivot(index='session_#', columns='depth_um', values='n_synapses')
        .fillna(0)
        .astype(int)
    )

    mouse_count_pivot = (
        session_depth_summary
        .pivot(index='session_#', columns='depth_um', values='n_mice')
        .fillna(0)
        .astype(int)
    )

    print('Detailed DMD-level synapse counts:')
    display(session_depth_counts.sort_values(['session_#', 'depth_um', 'subject_id', 'dmd']))

    print('Aggregated counts by recording session day and depth:')
    display(session_depth_summary)

    print('Synapse count pivot: rows = recording session_#, columns = depth_um')
    display(synapse_count_pivot)

    print('Mouse contribution pivot: rows = recording session_#, columns = depth_um')
    display(mouse_count_pivot)

if not missing_summary_df.empty:
    print('Sessions skipped or not loaded:')
    display(missing_summary_df.sort_values(['session_#', 'subject_id']))


In [ ]:
session_depth_counts[session_depth_counts['subject_id']==838410].sort_values('depth_um')